# 3장. Long-term Memory — Store 기본 CRUD

**Long-term Memory**는 대화방(thread_id)이 바뀌어도 유지되는 영구 저장소입니다.  
`InMemoryStore`를 사용해 데이터를 `namespace`(폴더) + `key`(파일명) 구조로 저장합니다.

| 개념 | 설명 | 예시 |
|:---|:---|:---|
| **namespace** | 폴더 경로 (튜플로 구성 — 충돌 방지) | `("user_001", "personal_assistant")` |
| **key** | 각 메모리 항목의 고유 식별자 | `"memory_001"` |

**주요 메서드:**
- `store.put(namespace, key, value)` — 저장
- `store.get(namespace, key)` — 특정 항목 조회
- `store.search(namespace)` — namespace 내 전체 검색

> 실제 서비스에서는 `InMemoryStore` 대신 PostgreSQL, Redis 등 영구 DB를 사용합니다.

In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

In [5]:
from langgraph.store.memory import InMemoryStore

# 1. Store 초기화
store = InMemoryStore()

user_id = "user_001"
application_context = "personal_assistant"

# 2. Namespace(폴더 경로) 설정
namespace = (user_id, application_context)


In [6]:
# 3. 첫 번째 데이터 저장
store.put(
    namespace,
    "memory_001",  # 이 데이터의 고유 식별자 (Key)
    {
        "facts": [
            "사용자는 커피보다 차를 선호함",
            "사용자는 매일 아침 6시에 일어남",
        ],
        "language": "Korean",
    },
)


In [7]:
# 4. 두 번째 데이터 저장
store.put(
    namespace,
    "memory_002",  # 이전 데이터가 덮어씌워지지 않도록 새로운 Key 지정
    {
        "facts": [
            "사용자는 그림 회화 작품을 좋아함",
            "빈센트 반 고흐의 작품을 특히 좋아함",
        ]
    },
)


In [ ]:
# 특정 Key 조회 (get)
item1 = store.get(namespace, "memory_001")
# print(item1)
print("1번 메모리:", item1.value)

item2 = store.get(namespace, "memory_002")
print("2번 메모리:", item2.value)

# Namespace 전체 검색 (search)
print("\n--- 전체 검색 결과 ---")
items = store.search(namespace)
# print(items)
for item in items:
    print(f"Key: {item.key} | Value: {item.value}")


1번 메모리: {'facts': ['사용자는 커피보다 차를 선호함', '사용자는 매일 아침 6시에 일어남'], 'language': 'Korean'}
2번 메모리: {'facts': ['사용자는 그림 회화 작품을 좋아함', '빈센트 반 고흐의 작품을 특히 좋아함']}

--- 전체 검색 결과 ---
[Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 커피보다 차를 선호함', '사용자는 매일 아침 6시에 일어남'], 'language': 'Korean'}, created_at='2026-03-22T10:21:45.319340+00:00', updated_at='2026-03-22T10:21:45.319345+00:00', score=None), Item(namespace=['user_001', 'personal_assistant'], key='memory_002', value={'facts': ['사용자는 그림 회화 작품을 좋아함', '빈센트 반 고흐의 작품을 특히 좋아함']}, created_at='2026-03-22T10:21:51.395306+00:00', updated_at='2026-03-22T10:21:51.395309+00:00', score=None)]
Key: memory_001 | Value: {'facts': ['사용자는 커피보다 차를 선호함', '사용자는 매일 아침 6시에 일어남'], 'language': 'Korean'}
Key: memory_002 | Value: {'facts': ['사용자는 그림 회화 작품을 좋아함', '빈센트 반 고흐의 작품을 특히 좋아함']}
